# LinkedIn writing habits: repeated-generation and cache smoke test

Writing a LinkedIn post from a news article means choosing a fact to lead with, explaining why a professional reader might care, and deciding where to stop. A useful post should remain faithful to its source while giving the reader a reason to spend time on it. The request can be short even when those choices are left entirely to the writer.

A model may return a different post on every request and still make those choices in much the same way. It might repeatedly begin with a contrast, put a short lesson near the end, and finish with a question. Comparing whole documents alone would miss that habit.

Consider an illustrative case: a company releases a new model, and someone asks an assistant to write a post about the announcement. Two responses can use different details from the article while sharing the same opening structure. Changing a product name or a number does not make that structure new. This example describes the measurement problem; it is not a model response from the study.

Now add one complication. The person can open a new chat for each attempt, or stay in the first chat and ask for another post. In the second case the assistant sees what it already wrote. That history can affect both the angle of the next post and how often phrases recur. A study needs to say which situation it measures.

There is also a difference between generating a new answer and processing the same input again. A provider can reuse saved calculations for an unchanged beginning of a conversation while still generating a fresh answer. This is called prompt-prefix caching. It reduces repeated input work; it does not mean returning a previously saved post.

An illustrative calculation makes the distinction concrete. If a request processes 1,000 input tokens and 900 are reported as cache reads, its read fraction is 900 / 1,000 = 90%. Two requests should be combined by adding their cached tokens and their input tokens before dividing. Averaging percentages can be misleading when the requests have different lengths.

Conversation history makes later requests longer even when the new message is only three words. Earlier user messages and assistant answers still enter the model's context. A higher cache-hit fraction therefore does not by itself establish a smaller bill. We need the total input, new cache writes, output and any reported cost too.

This smoke test asks two engineering questions before a larger writing study: do repeated requests yield distinct usable post bodies, and can the harness measure cache reuse accurately for fresh sessions and continued conversations? It also checks that openings, endings, formatting and editorial notes can be counted without mixing them together.


The two conditions returned **40 readable responses out of 40 planned calls**, containing five distinct post bodies per model per condition. No tool calls were observed. Continued conversations used the original article prompt once, followed by exactly `Generate another post.` on turns 2-5.

For GPT-6.1 Terra, the cached-input fraction on requests 2-5 rose from 49.46% in fresh sessions to 98.26% in one conversation. Sol rose from 0% to 48.39%. Both Claude models already read nearly all input from cache in fresh requests; continued turns wrote new history and read about 89% of their input from cache. These are measurements on one article in one ordered run, not general provider guarantees.

The main caveat is that this is an engineering smoke test. The source topic, execution order and previous cache state are not independently controlled. Distinct full posts do not establish stylistic variety, factual correctness or quality. The 20-topic, 400-post main study has not been run.


## Contents

1. [Setup and configuration](#1-setup-and-configuration)
2. [Source and exact prompts](#2-source-and-exact-prompts)
3. [Distinct outputs](#3-distinct-outputs)
4. [Provider cache accounting](#4-provider-cache-accounting)
5. [Prose and formatting inventory](#5-prose-and-formatting-inventory)
6. [Worked cases and parser repairs](#6-worked-cases-and-parser-repairs)
7. [What this test can establish](#7-what-this-test-can-establish)
8. [Spend and reproduction](#8-spend-and-reproduction)
9. [Future work and further reading](#9-future-work-and-further-reading)

```mermaid
flowchart TD
    A["1 complete article<br/>2,318 words; frozen local input"] --> B["Same initial instruction<br/>4 models, high effort"]
    B --> C["Fresh condition<br/>5 new sessions per model; 20 posts"]
    B --> D["Conversation condition<br/>1 session per model; 5 turns; 20 posts"]
    D --> E["Turns 2-5<br/>Generate another post."]
    C --> F["Raw CLI events and final text<br/>40 response records"]
    E --> F
    F --> G["Per-turn usage and hashes<br/>111 fixed prose candidates; formatting"]
    G --> H["Tables, examples and figures<br/>offline replay from saved responses"]
```

One article travels through both branches. In the fresh branch its complete text is sent on every request. In the conversation branch it is sent only on the first turn; the CLI restores the same conversation before each short follow-up. The provider returns text and usage records. The notebook checks both without asking a model to judge its own writing.


## 1. Setup and configuration

All four models run at high effort through their CLIs. No generation HTTP API or SDK is used. The optional base instruction is `You are a helpful assistant.` The full commands and CLI versions are saved in [the preflight record](data/preflight.json). The supported model IDs are fixed, but the CLIs do not expose immutable dated revisions, so this is not a claim of reproducibility under frozen weights.

| Model | Requested version / immutable pin | Provider and call | Size / precision | Output settings | Why included |
|---|---|---|---|---|---|
| GPT-6.1 Sol | `gpt-6.1-sol`; no immutable dated revision exposed | OpenAI, `codex exec` | Not published | High effort; default supported sampling; optional base prompt replaced by neutral line | Requested member of the GPT-6.1 comparison |
| GPT-6.1 Terra | `gpt-6.1-terra`; no immutable dated revision exposed | OpenAI, `codex exec` | Not published | High effort; default supported sampling; optional base prompt replaced by neutral line | Requested member of the GPT-6.1 comparison |
| Claude Opus 5.5 | `claude-opus-5-5`; no immutable dated revision exposed | Anthropic, `claude -p` | Not published | High effort; default supported sampling; optional base prompt replaced by neutral line | Requested member of the Claude 5.5 comparison |
| Claude Sonnet 5.5 | `claude-sonnet-5-5`; no immutable dated revision exposed | Anthropic, `claude -p` | Not published | High effort; default supported sampling; optional base prompt replaced by neutral line | Requested member of the Claude 5.5 comparison |

The table identifies the actual requested models and what is held constant. Equal named effort does not imply equal hidden computation. Codex 0.160.1 and Claude Code 2.1.296 ran outside the repository, with web search, command execution, browser/computer/image operations, MCP, optional skills, hooks, plugins and multi-agent operation disabled through supported controls. Child processes did not receive API-key or parent CLI-session environment variables. Existing CLI authentication was retained.

Claude init events report empty tools, MCP servers and skills. Its three built-in plugin inventory labels remain even in safe mode. Codex debug output still contains mandatory runtime skill-catalogue/team directives and environment context despite discovery/feature disables; no repository writing rules were found. Codex JSONL does not expose the entire advertised tool catalogue. Therefore zero observed tool calls and disabled features are verified; a completely identical bare provider system prompt is not. [Protocol history](data/PROTOCOL.md) records this boundary.

The code below defines all helpers inside the notebook. Default execution only replays saved responses. Live collection is an explicit opt-in with small call limits.

In [1]:
"""Notebook 40 collection and deterministic replay. No model APIs or SDKs."""
import argparse
from collections import Counter
import csv
import hashlib
import itertools
import json
import os
from pathlib import Path
import random
import re
import shutil
import subprocess
import tempfile
import time
import unicodedata
from datetime import datetime, timezone
from difflib import SequenceMatcher

ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
DATA = ROOT / 'data'
MODELS = ['gpt-6.1-sol', 'gpt-6.1-terra', 'claude-opus-5-5', 'claude-sonnet-5-5']
EFFORT = 'high'
N_ARTICLES = 1
N_REPEATS = 5
MAX_NEW_CALLS = 20
TIMEOUT_SECONDS = 240
PROTOCOL = 'smoke-v2-high-cache-blocked'
SOURCE_URL = 'https://huggingface.co/blog/mistral-large4'
PROMPT_TEMPLATE = 'Write a LinkedIn post based on the following article.\n\n<article>\n{article}\n</article>'
FOLLOWUP_PROMPT = 'Generate another post.'
NEUTRAL = 'You are a helpful assistant.\n'


def canonical(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(',', ':'))


def sha(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def save(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


def load(path):
    return json.loads(path.read_text(encoding='utf-8'))


def normalise(text):
    text = unicodedata.normalize('NFKC', text).lower().replace('\u2019', "'")
    return ' '.join(text.split())


def words(text):
    return re.findall(r"\b\w+(?:['\u2019-]\w+)*\b", text.lower())


def freeze_source():
    """Whole article body, locally retained; URL/hash are the public provenance."""
    import requests
    from bs4 import BeautifulSoup
    target = DATA / 'local_inputs' / 'mistral-large4.txt'
    if target.exists():
        assert sha(target.read_text(encoding='utf-8')) == load(DATA / 'article.json')['text_sha256']
        return
    response = requests.get(SOURCE_URL, timeout=45)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')
    wrapper = soup.select_one('.blog-content')
    body = wrapper.select_one('div.relative.overflow-clip')
    assert wrapper and body, 'Article extraction selector changed'
    # Remove the duplicate interactive contents widget, retain the written contents.
    for widget in body.select('.SVELTE_HYDRATER'):
        widget.decompose()
    chunks = [wrapper.h1.get_text(' ', strip=True), 'Published October 29, 2026']
    for child in body.find_all(recursive=False):
        if child.name == 'pre':
            chunks.append(child.get_text('', strip=False).strip())
        else:
            chunks.append(child.get_text(' ', strip=True))
    article = '\n\n'.join(x for x in chunks if x).strip() + '\n'
    assert len(article) > 10000 and 'Mistral Large 4' in article
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(article, encoding='utf-8')
    save(DATA / 'article.json', {
        'item_id': 'smoke-mistral-large4', 'url': SOURCE_URL,
        'title': chunks[0], 'publication_date': '2026-10-29',
        'retrieved_at': datetime.now(timezone.utc).isoformat(),
        'text_sha256': sha(article), 'characters': len(article), 'words': len(words(article)),
        'raw_html_sha256': hashlib.sha256(response.content).hexdigest(),
        'local_input': 'data/local_inputs/mistral-large4.txt',
        'redistribution': 'Full source text retained locally and git-ignored; no redistribution licence assumed.',
        'extraction': 'Title, publication date and complete article body; duplicate interactive contents widget removed. Code retained as text. No linked pages fetched into input.',
        'role': 'Smoke only; exclude this topic from a later confirmatory sample.'})


def freeze_lexicon():
    target = DATA / 'prose_lexicon.json'
    if target.exists():
        return
    groups = {
        'vocabulary': 'delve|tapestry|testament|intricate|meticulous|pivotal|crucial|vital|robust|seamless|leverage|harness|unlock|elevate|empower|foster|underscore|showcase|realm|landscape|navigate|journey|embark|game-changer|cutting-edge|groundbreaking|transformative|holistic|multifaceted|nuanced|comprehensive|invaluable|vibrant|boasts|notably|additionally|furthermore|moreover',
        'stock_phrases': "plays a key role|a wide range of|a myriad of|valuable insights|it's worth noting|it's important to note|let's dive in|sets the stage|marks a pivotal moment|in an evolving landscape|in today's fast-paced world|when it comes to|studies show|experts agree|it is widely known|serves as|stands as|functions as|may potentially|could possibly|it seems that|state-of-the-art",
        'linkedin_candidates': "this changes everything|the future is|the future of|here's why|here's the thing|here's what|here's the catch|let that sink in|read that again|think about that|a reminder that|what this means|why this matters|the real question|the bigger picture|the bottom line|my takeaway|my take|key takeaway|key takeaways|in conclusion|in summary|at the end of the day|the takeaway|the result|the lesson|the shift|the opportunity|food for thought|what do you think|what's your take|agree or disagree|curious to hear|would love to hear|let me know|drop a comment|share your thoughts|thoughts|certainly|great question|i hope this helps",
    }
    entries = []
    for category, phrases in groups.items():
        for phrase in phrases.split('|'):
            escaped = re.escape(phrase).replace(r'\ ', r'\s+')
            # Stem selected vocabulary words, while keeping ordinary boundary matching.
            tails = {'harness': r'(?:es|ed|ing)?', 'unlock': r'(?:s|ed|ing)?'}
            regex = r'\b' + escaped + tails.get(phrase, '') + r'\b'
            if phrase in ['delve', 'leverage', 'underscore', 'showcase']:
                regex = r'\b' + escaped[:-1] + r'(?:e|es|ed|ing)\b'
            entries.append({'id': phrase.replace(' ', '_'), 'label': phrase, 'category': category,
                            'regex': regex, 'status': 'fixed_before_smoke'})
    patterns = {
        'not_x_but_y': r"\bnot\b[^.!?\n]{1,140}\bbut\b",
        'isnt_x_its_y': r"\b(?:isn't|is not)\b[^.!?\n]{1,140}(?:[.!?]\s+|[,;:]\s*)?(?:it's|it is)\b",
        'not_just_x': r'\bnot (?:just|only)\b',
        'trailing_significance': r',\s*(?:highlighting|underscoring|showcasing|demonstrating|reinforcing|signaling)\b',
        'teaser_question': r"\b(?:the (?:result|lesson|takeaway|catch|question)|what changed|why it matters)\?",
        'colon_reveal': r"\b(?:here's (?:the|what|why)|the (?:result|lesson|takeaway|catch|question))[^\n.!?]{0,60}:",
        'recap_signal': r'\b(?:in conclusion|in summary|overall|key takeaways?)\b',
        'vague_authority': r'\b(?:studies show|experts agree|research suggests|it is widely known)\b',
        'stacked_hedge': r'\b(?:may potentially|could possibly|might perhaps)\b',
        'significance_claim': r'\b(?:significantly|dramatically|substantially|remarkably)\b',
    }
    for label, regex in patterns.items():
        entries.append({'id': label, 'label': label, 'category': 'structure_candidate',
                        'regex': regex, 'status': 'fixed_before_smoke',
                        'caution': 'Candidate span requiring contextual review, not an automatic defect.'})
    save(target, {'version': 1, 'frozen_at': datetime.now(timezone.utc).isoformat(),
                  'sources': ['Repository AGENTS.md editorial watchlist',
                              'https://arxiv.org/abs/2406.07016', 'https://arxiv.org/abs/2412.11385'],
                  'scope': 'Papers motivate vocabulary analysis; the repository supplies editorial candidates. LinkedIn-specific phrases and regexes are investigator hypotheses. No entry is an AI authorship detector.',
                  'entries': entries})


def clean_env():
    return {k: v for k, v in os.environ.items()
            if not k.startswith(('CODEX_', 'CLAUDE_')) and k not in ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY']}


def isolation_dir():
    # Same path for all repetitions: changing cwd can invalidate the prompt prefix.
    directory = Path(tempfile.gettempdir()) / 'linkedin-40-isolation'
    directory.mkdir(exist_ok=True)
    (directory / 'neutral.txt').write_text(NEUTRAL, encoding='utf-8')
    (directory / 'empty-mcp.json').write_text('{"mcpServers":{}}\n', encoding='utf-8')
    return directory


def codex_options():
    directory = isolation_dir()
    flags = ['--ignore-user-config', '--ignore-rules', '--skip-git-repo-check', '--ephemeral', '--strict-config']
    for feature in ['shell_tool', 'unified_exec', 'apps', 'plugins', 'multi_agent', 'browser_use',
                    'computer_use', 'image_generation', 'code_mode', 'code_mode_host', 'shell_snapshot',
                    'hooks', 'memories', 'skill_search', 'tool_suggest', 'sleep_tool', 'goals', 'view_image']:
        flags += ['--disable', feature]
    flags += ['--enable', 'skip_host_skill_discovery']
    for key, value in {'web_search': '"disabled"', 'model_reasoning_effort': '"high"',
                       'project_doc_max_bytes': '0', 'personality': '"none"',
                       'developer_instructions': '""',
                       'model_instructions_file': json.dumps((directory / 'neutral.txt').as_posix())}.items():
        flags += ['-c', f'{key}={value}']
    return flags


def command(model, conversation=False, session_id=None):
    if model.startswith('gpt'):
        options = codex_options()
        if conversation:
            options.remove('--ephemeral')
        return [shutil.which('codex.cmd') or shutil.which('codex'), 'exec', *options,
                '-m', model, '--json', *(['resume', session_id] if session_id else []), '-']
    cmd = [shutil.which('claude'), '-p', '--model', model, '--effort', EFFORT,
            '--tools', '', '--safe-mode', '--disable-slash-commands', '--strict-mcp-config', '--mcp-config',
            str(isolation_dir() / 'empty-mcp.json'), '--disallowedTools', 'mcp__*',
            '--no-session-persistence', '--system-prompt', NEUTRAL.strip(),
            '--output-format', 'stream-json', '--verbose', '--max-budget-usd', '1.00']
    if conversation:
        cmd.remove('--no-session-persistence')
    if session_id:
        cmd += ['--resume', session_id]
    return cmd


def json_events(stdout):
    events = []
    for line in stdout.splitlines():
        try:
            value = json.loads(line)
            if isinstance(value, dict):
                events.append(value)
        except json.JSONDecodeError:
            pass
    return events


def parse_result(model, stdout):
    events = json_events(stdout)
    text_parts, tool_events, catalogues, cli_warnings = [], [], [], []
    usage, session, resolved, reported_cost, completed = {}, None, None, None, False
    for e in events:
        if model.startswith('gpt'):
            session = e.get('thread_id', session)
            if e.get('type') == 'item.completed':
                item = e.get('item', {})
                if item.get('type') == 'agent_message':
                    text_parts.append(item.get('text', ''))
                elif item.get('type') == 'error':
                    cli_warnings.append(item)
                elif item.get('type') not in ['reasoning']:
                    tool_events.append(item)
            if e.get('type') == 'turn.completed':
                usage, completed = e.get('usage', {}), True
        else:
            session = e.get('session_id', session)
            if e.get('type') == 'system' and e.get('subtype') == 'init':
                catalogues.append({'tools': e.get('tools'), 'mcp_servers': e.get('mcp_servers'),
                                   'skills': e.get('skills'), 'plugins': e.get('plugins')})
                resolved = e.get('model', resolved)
            if e.get('type') == 'assistant':
                for block in e.get('message', {}).get('content', []):
                    if block.get('type') == 'tool_use':
                        tool_events.append(block)
            if e.get('type') == 'result':
                usage = e.get('usage', {})
                text_parts = [e.get('result', '')]
                reported_cost = e.get('total_cost_usd')
                completed = e.get('subtype') == 'success' and not e.get('is_error', False)
    text = '\n\n'.join(text_parts).strip()
    if model.startswith('gpt'):
        total = usage.get('input_tokens')
        details = usage.get('input_tokens_details', {})
        reads = usage.get('cached_input_tokens', details.get('cached_tokens'))
        writes = usage.get('cache_write_input_tokens', details.get('cache_write_tokens'))
        uncached = total - reads - writes if all(isinstance(x, int) for x in [total, reads, writes]) else None
    else:
        uncached = usage.get('input_tokens')
        reads, writes = usage.get('cache_read_input_tokens'), usage.get('cache_creation_input_tokens')
        total = sum([uncached, reads, writes]) if all(isinstance(x, int) for x in [uncached, reads, writes]) else None
    return {'text': text, 'completed': completed, 'session_id': session,
            'reported_model': resolved, 'raw_usage': usage, 'input_total': total,
            'cache_read_tokens': reads, 'cache_write_tokens': writes, 'uncached_input_tokens': uncached,
            'output_tokens': usage.get('output_tokens'), 'reported_cost_usd': reported_cost,
            'reported_session_cost_usd': reported_cost,
            'cache_read_fraction': reads / total if isinstance(reads, int) and total else None,
            'tool_events': tool_events, 'catalogues': catalogues, 'cli_warnings': cli_warnings,
            'raw_sha256': sha(stdout), 'text_sha256': sha(text), 'normalised_sha256': sha(normalise(text))}


def preflight():
    """No generation: inspect prompt context and validate supported flags locally."""
    cmd = [shutil.which('codex.cmd') or shutil.which('codex'), 'debug', 'prompt-input']
    flags = codex_options()
    # These options belong to exec rather than debug.
    flags = [x for x in flags if x not in ['--ignore-user-config', '--ignore-rules', '--skip-git-repo-check', '--ephemeral', '--strict-config']]
    result = subprocess.run(cmd + flags + ['PROBE_ONLY_NO_GENERATION'], cwd=isolation_dir(), env=clean_env(),
                            capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=30)
    assert result.returncode == 0, result.stderr
    messages = json.loads(result.stdout)
    strings = [b.get('text', '') for m in messages for b in m.get('content', []) if isinstance(b, dict)]
    forbidden = ['# Instructions for AI agents', 'No em dashes.', 'Write a LinkedIn post', 'Use plain verbs.']
    assert not any(f in '\n'.join(strings) for f in forbidden), 'Writing/repository context contamination'
    save(DATA / 'preflight.json', {'protocol': PROTOCOL, 'checked_at': datetime.now(timezone.utc).isoformat(),
          'prompt_context_sha256': sha(result.stdout),
          'context_blocks': [{'role': m.get('role'), 'characters': len(canonical(m.get('content', [])))} for m in messages],
          'residual_context': 'Codex debug exposes runtime-injected skills catalogue, team directives and environment context despite feature disables. No repository writing rules observed. These platform-level instructions cannot be claimed removed.',
          'cli_versions': {name: subprocess.run([shutil.which(name + '.cmd') or shutil.which(name), '--version'], capture_output=True, text=True).stdout.strip() for name in ['codex', 'claude']},
          'commands': {model: command(model) for model in MODELS},
          'tool_catalogue_validation': 'Claude init catalogue checked per call; Codex disabled features plus event trace checked. Codex exec does not expose a complete advertised tool catalogue in JSONL.'})


def collect(repeats=N_REPEATS, max_new=MAX_NEW_CALLS, conversation=False):
    assert 1 <= repeats <= 5 and 0 <= max_new <= MAX_NEW_CALLS
    freeze_source()
    freeze_lexicon()
    preflight()
    article = (DATA / 'local_inputs' / 'mistral-large4.txt').read_text(encoding='utf-8')
    prompt = PROMPT_TEMPLATE.format(article=article)
    lexicon_hash = sha((DATA / 'prose_lexicon.json').read_text(encoding='utf-8'))
    model_order = MODELS.copy()
    random.Random(20261101).shuffle(model_order)
    n_new = 0
    for model in model_order:
        previous_session = None
        previous_session_cost = 0.0
        request = {'protocol': PROTOCOL, 'model': model, 'effort': EFFORT, 'prompt_sha256': sha(prompt),
                   'source_sha256': sha(article), 'system_sha256': sha(NEUTRAL),
                   'lexicon_sha256': lexicon_hash, 'command': command(model)}
        for replicate in range(1, repeats + 1):
            cmd = command(model, conversation, previous_session)
            request['command'] = cmd
            turn_prompt = FOLLOWUP_PROMPT if conversation and replicate > 1 else prompt
            request['prompt_sha256'] = sha(turn_prompt)
            if conversation:
                request['history_condition'] = 'same_conversation_initial_article_then_generate_another_post_only'
                request['turn_prompt_kind'] = 'followup_only' if replicate > 1 else 'initial_article'
            request_hash = sha(canonical(request))
            slot = DATA / ('conversation' if conversation else 'smoke') / model / f'{replicate:02d}-{request_hash[:12]}'
            if slot.with_suffix('.json').exists():
                saved = load(slot.with_suffix('.json'))
                assert saved['status'] == 'ok', 'Saved failed slot requires diagnosis, not silent replay'
                previous_session = saved['session_id'] if conversation else None
                if conversation and not model.startswith('gpt'):
                    previous_session_cost = saved.get('reported_session_cost_usd', saved.get('reported_cost_usd'))
                continue
            if slot.with_suffix('.started.json').exists():
                raise RuntimeError('Unresolved started request: reconcile its provider result before generating again')
            if conversation and any(int(p.name[:2]) > replicate for p in slot.parent.glob('*.json') if p.name[:2].isdigit()):
                raise RuntimeError('Conversation has later saved turns: do not fill a gap by appending to changed history')
            if n_new >= max_new:
                raise RuntimeError('New-call budget exhausted')
            start = time.monotonic()
            started_at = datetime.now(timezone.utc).isoformat()
            print(f'START {model} repeat {replicate}', flush=True)
            slot.parent.mkdir(parents=True, exist_ok=True)
            save(slot.with_suffix('.started.json'), {'requested_model': model, 'replicate': replicate,
                 'request': request.copy(), 'started_at': started_at, 'status': 'started_usage_not_yet_available'})
            try:
                result = subprocess.run(cmd, input=turn_prompt, cwd=isolation_dir(), env=clean_env(),
                                        capture_output=True, text=True, encoding='utf-8', errors='replace',
                                        timeout=TIMEOUT_SECONDS)
                stdout, stderr, code = result.stdout, result.stderr, result.returncode
            except subprocess.TimeoutExpired as exc:
                stdout = (exc.stdout or b'').decode('utf-8', errors='replace') if isinstance(exc.stdout, bytes) else (exc.stdout or '')
                stderr, code = 'CLI timeout; no automatic content retry', -1
            parsed = parse_result(model, stdout)
            if conversation and not model.startswith('gpt') and parsed['reported_session_cost_usd'] is not None:
                parsed['reported_cost_usd'] = parsed['reported_session_cost_usd'] - previous_session_cost
                previous_session_cost = parsed['reported_session_cost_usd']
            parsed.update({'requested_model': model, 'replicate': replicate, 'request': request,
                           'request_sha256': request_hash, 'started_at': started_at,
                           'seconds': time.monotonic() - start, 'exit_code': code,
                           'local_replay': False})
            parsed['status'] = ('ok' if code == 0 and parsed['completed'] and parsed['text'] else 'unreadable_or_failed')
            if parsed['tool_events']:
                parsed['status'] = 'tool_protocol_violation'
            for cat in parsed['catalogues']:
                if any(cat.get(k) for k in ['tools', 'mcp_servers']):
                    parsed['status'] = 'catalogue_protocol_violation'
            slot.with_suffix('.jsonl').write_text(stdout, encoding='utf-8')
            slot.with_suffix('.stderr.txt').write_text(stderr, encoding='utf-8')
            save(slot.with_suffix('.json'), parsed)
            slot.with_suffix('.started.json').unlink()
            n_new += 1
            print(f"DONE {model} repeat {replicate}: {parsed['status']}; input={parsed['input_total']}; cache_read={parsed['cache_read_tokens']}; cache_write={parsed['cache_write_tokens']}; seconds={parsed['seconds']:.1f}", flush=True)
            if parsed['status'] != 'ok':
                raise RuntimeError('Smoke gate failed; inspect saved evidence before any more calls')
            if conversation:
                if previous_session is not None and previous_session != parsed['session_id']:
                    raise RuntimeError('Conversation changed session ID on resume')
                previous_session = parsed['session_id']


def split_response(text):
    """Conservative literal wrapper/note extraction; keep all raw text separately."""
    body, annotation, preface = text.strip(), '', ''
    parts = re.split(r'\n\s*---\s*\n', body)
    if len(parts) > 1:
        tail = parts[-1].strip()
        plain = tail.lstrip('*_ ').lower()
        if re.match(r'(?:note\b|one note\b|the article\b|i left out\b|before posting\b)', plain):
            annotation = tail
            body = '\n\n---\n\n'.join(parts[:-1]).strip()
    lines = body.splitlines()
    if lines and re.fullmatch(r'\*{0,2}linkedin post:\*{0,2}', lines[0].strip(), re.I):
        preface = lines.pop(0)
        while lines and (not lines[0].strip() or lines[0].strip() == '---'):
            lines.pop(0)
        body = '\n'.join(lines).strip()
    return {'body': body, 'preface': preface, 'annotation': annotation}


def sentences(text):
    # An emoji after a final question belongs to that sentence, not a new one.
    return re.split(r'''(?<=[.!?])\s+(?=["\u201c'\u2018*]*[A-Za-z0-9])''', text.strip()) if text.strip() else []


def zones(text):
    raw = text
    segments = split_response(text)
    text = segments['body']
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    substantive = [p for p in paragraphs if not all(w.startswith('#') for w in p.split())]
    ss_all = sentences(text)
    starts, endings = [], []
    for p in substantive:
        ss = sentences(p)
        starts.extend(ss[1:])
    for p in substantive[1:]:
        starts.append(p)
    for p in substantive[:-1]:
        endings.append(sentences(p)[-1])
    return {'whole_response': raw, 'editorial_note': segments['annotation'],
            'whole': text, 'opening_sentence': ss_all[0] if ss_all else '',
            'opening_paragraph': paragraphs[0] if paragraphs else '',
            'opening_30_words': ' '.join(text.split()[:30]),
            'transition_starts': '\n'.join(' '.join(x.split()[:12]) for x in starts),
            'point_endings': '\n'.join(endings),
            'conclusion': substantive[-1] if substantive else '',
            'closing': sentences(substantive[-1])[-1] if substantive else ''}


def formatting(text):
    segments = split_response(text)
    text = segments['body']
    paragraphs = [p for p in re.split(r'\n\s*\n', text.strip()) if p]
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return {'words': len(words(text)), 'characters': len(text), 'paragraphs': len(paragraphs),
            'response_preface': bool(segments['preface']), 'editorial_note': bool(segments['annotation']),
            'editorial_note_words': len(words(segments['annotation'])),
            'nonempty_lines': len(lines), 'blank_lines': sum(not l.strip() for l in text.splitlines()),
            'short_lines_le_8_words': sum(len(words(l)) <= 8 for l in lines),
            'bullets': len(re.findall(r'^\s*[-*\u2022]\s+', text, re.M)),
            'numbered_items': len(re.findall(r'^\s*\d+[.)]\s+', text, re.M)),
            'hashtags': len(re.findall(r'(?<!\w)#[\w]+', text)),
            'headings': len(re.findall(r'^\s*#{1,6}\s+', text, re.M)),
            'bold_spans': len(re.findall(r'\*\*.+?\*\*|__.+?__', text)),
            'links': len(re.findall(r'https?://\S+', text)),
            'em_dashes': text.count('\u2014'), 'colons': text.count(':'),
            'questions': text.count('?'), 'exclamations': text.count('!'),
            'ellipses': text.count('...') + text.count('\u2026'),
            'emoji_codepoints': sum(0x1F300 <= ord(c) <= 0x1FAFF or 0x2600 <= ord(c) <= 0x27BF for c in text),
            'emoji_prefixed_lines': len(re.findall(r'^\s*[\U0001F300-\U0001FAFF\u2600-\u27BF]', text, re.M)),
            'opening_question': '?' in zones(text)['opening_sentence'],
            'closing_question': bool(re.search(r'\?[\s\W]*$', zones(text)['closing']))}


def write_csv(path, rows):
    if not rows:
        return
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def refresh_accounting(arm):
    """Convert cumulative resumed-session counters to per-invocation increments."""
    for model in MODELS:
        previous_usage, previous_cost = {}, 0.0
        for path in sorted((DATA / arm / model).glob('*.json')):
            if '.started.' in path.name:
                continue
            record = load(path)
            parsed = parse_result(model, path.with_suffix('.jsonl').read_text(encoding='utf-8'))
            native = parsed['raw_usage'].copy()
            if arm == 'conversation' and model.startswith('gpt'):
                delta = {k: v - previous_usage.get(k, 0) for k, v in native.items() if isinstance(v, int)}
                assert all(v >= 0 for v in delta.values()), 'Cumulative usage decreased'
                parsed['raw_session_usage'] = native
                parsed['per_turn_usage'] = delta
                parsed['input_total'] = delta.get('input_tokens')
                parsed['cache_read_tokens'] = delta.get('cached_input_tokens')
                parsed['cache_write_tokens'] = delta.get('cache_write_input_tokens')
                parsed['output_tokens'] = delta.get('output_tokens')
                parsed['uncached_input_tokens'] = parsed['input_total'] - parsed['cache_read_tokens'] - parsed['cache_write_tokens']
                parsed['cache_read_fraction'] = parsed['cache_read_tokens'] / parsed['input_total']
                parsed['usage_scope'] = 'Codex cumulative session counters differenced by turn; raw counters retained'
                previous_usage = native
            else:
                parsed['usage_scope'] = 'Per-invocation usage'
            if arm == 'conversation' and not model.startswith('gpt'):
                total = parsed['reported_session_cost_usd']
                parsed['reported_cost_usd'] = total - previous_cost if total is not None else None
                previous_cost = total
                parsed['usage_scope'] += '; Claude cumulative reported session cost differenced by turn'
            record.update(parsed)
            save(path, record)


def analyse(arm='smoke'):
    assert arm in ['smoke', 'conversation']
    records = [load(p) for p in sorted((DATA / arm).glob('*/*.json'))]
    prefix = '' if arm == 'smoke' else 'conversation_'
    lexicon = load(DATA / 'prose_lexicon.json')['entries']
    source_path = DATA / 'local_inputs' / 'mistral-large4.txt'
    source = normalise(source_path.read_text(encoding='utf-8')) if source_path.exists() else None
    overlap_path = DATA / 'source_overlap.json'
    overlaps = load(overlap_path) if overlap_path.exists() else {'lexicon': {}, 'candidate_sha256': {}}
    rows, pairs, phrase_rows, format_rows, candidates = [], [], [], [], []
    for model in MODELS:
        rs = sorted([r for r in records if r['requested_model'] == model], key=lambda r: r['replicate'])
        valid = [r for r in rs if r['status'] == 'ok']
        for a, b in itertools.combinations(valid, 2):
            wa, wb = words(a['text']), words(b['text'])
            na = set(zip(*(wa[i:] for i in range(3))))
            nb = set(zip(*(wb[i:] for i in range(3))))
            pairs.append({'model': model, 'repeat_a': a['replicate'], 'repeat_b': b['replicate'],
                          'exact_duplicate': a['text_sha256'] == b['text_sha256'],
                          'normalised_duplicate': a['normalised_sha256'] == b['normalised_sha256'],
                          'word_sequence_similarity': SequenceMatcher(None, wa, wb, autojunk=False).ratio(),
                          'trigram_jaccard': len(na & nb) / len(na | nb) if na | nb else 1.0})
        reads = [r['cache_read_tokens'] for r in valid]
        totals = [r['input_total'] for r in valid]
        warm = valid[1:]
        all_known = bool(valid) and all(isinstance(x, int) for x in reads + totals)
        warm_known = bool(warm) and all(isinstance(r[k], int) for r in warm for k in ['cache_read_tokens', 'input_total'])
        rows.append({'model': model, 'effort': EFFORT, 'attempted': len(rs), 'readable': len(valid),
                     'unique_exact': len({r['text_sha256'] for r in valid}),
                     'unique_normalised': len({r['normalised_sha256'] for r in valid}),
                     'unique_post_normalised': len({normalise(split_response(r['text'])['body']) for r in valid}),
                     'cache_read_tokens': sum(reads) if all_known else None,
                     'input_total_tokens': sum(totals) if all_known else None,
                     'cache_read_fraction_all': sum(reads) / sum(totals) if all_known and sum(totals) else None,
                     'cache_read_fraction_repeats_2_to_5': sum(r['cache_read_tokens'] for r in warm) / sum(r['input_total'] for r in warm) if warm_known else None,
                     'cache_write_tokens': sum(r['cache_write_tokens'] for r in valid) if valid and all(isinstance(r['cache_write_tokens'], int) for r in valid) else None,
                     'reported_cost_usd': sum(r['reported_cost_usd'] for r in valid) if valid and all(isinstance(r['reported_cost_usd'], (int,float)) for r in valid) else None,
                     'seconds': sum(r['seconds'] for r in rs),
                     'tool_events': sum(len(r['tool_events']) for r in rs)})
        for r in valid:
            format_rows.append({'model': model, 'replicate': r['replicate'], **formatting(r['text'])})
        for zone in zones(''):
            document_counts = Counter()
            for record in valid:
                tokens = words(normalise(zones(record['text'])[zone]))
                seen = set()
                for n in range(2, 9):
                    seen.update(' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1))
                document_counts.update(seen)
            source_tokens = ' ' + ' '.join(words(source)) + ' ' if source is not None else None
            for phrase, count in document_counts.items():
                if count >= 2:
                    if source_tokens is not None:
                        overlaps['candidate_sha256'][sha(phrase)] = (' ' + phrase + ' ') in source_tokens
                    candidates.append({'model': model, 'zone': zone, 'phrase': phrase,
                        'words': len(phrase.split()), 'posts_containing': count,
                        'readable_posts': len(valid), 'distinct_topics': 1,
                        'present_in_source': overlaps['candidate_sha256'].get(sha(phrase)),
                        'status': 'exploratory_one_topic_smoke'})
            for entry in lexicon:
                regex = re.compile(entry['regex'], re.I)
                texts = [normalise(zones(r['text'])[zone]) for r in valid]
                counts = [len(regex.findall(text)) for text in texts]
                denominator = sum(len(words(text)) for text in texts)
                if source is not None:
                    overlaps['lexicon'][entry['id']] = len(regex.findall(source))
                phrase_rows.append({'model': model, 'zone': zone, 'phrase_id': entry['id'],
                    'category': entry['category'], 'occurrences': sum(counts),
                    'posts_containing': sum(c > 0 for c in counts), 'readable_posts': len(valid),
                    'post_fraction': sum(c > 0 for c in counts) / len(valid) if valid else None,
                    'occurrences_per_1000_words': 1000 * sum(counts) / denominator if denominator else None,
                    'source_occurrences': overlaps['lexicon'].get(entry['id'])})
    write_csv(DATA / f'{arm}_summary.csv', rows)
    write_csv(DATA / f'{prefix}pairwise_similarity.csv', pairs)
    write_csv(DATA / f'{prefix}prose_counts.csv', phrase_rows)
    write_csv(DATA / f'{prefix}formatting.csv', format_rows)
    save(overlap_path, overlaps)
    write_csv(DATA / f'{prefix}phrase_candidates.csv', sorted(candidates, key=lambda r:(r['model'],r['zone'],-r['posts_containing'],-r['words'],r['phrase'])))
    write_csv(DATA / f'{prefix}provider_cache_usage.csv', [{k:r.get(k) for k in [
        'requested_model','replicate','status','input_total','uncached_input_tokens','cache_read_tokens',
        'cache_write_tokens','cache_read_fraction','output_tokens','reported_cost_usd','seconds','session_id']} for r in records])
    save(DATA / f'{arm}_summary.json', {'protocol': PROTOCOL, 'condition': arm, 'models': rows,
        'gates': {'twenty_successful': sum(r['status'] == 'ok' for r in records) == 20,
                  'five_unique_per_model': all(r['unique_normalised'] == 5 for r in rows),
                  'five_unique_post_bodies_per_model': all(r['unique_post_normalised'] == 5 for r in rows),
                  'no_observed_tool_events': not any(r['tool_events'] for r in records),
                  'prompts_match_condition': (len({r['request']['prompt_sha256'] for r in records}) == 1 if arm == 'smoke' else
                      all(r['request']['prompt_sha256'] == (sha(FOLLOWUP_PROMPT) if r['replicate'] > 1 else records[0]['request']['prompt_sha256']) for r in records)),
                  'sessions_match_condition': (len({r['session_id'] for r in records}) == len(records) if arm == 'smoke' else
                      all(len({r['session_id'] for r in records if r['requested_model'] == model}) == 1 for model in MODELS))},
        'interpretation': 'Smoke is an engineering check on one topic, not a model ranking or evidence of general family habits. Provider caching reuses input computation, not output text. Unknown usage fields remain null.'})
    print(json.dumps(rows, indent=2))
    return rows



In [2]:
# Replay only: no CLI or network call is made.
fresh_rows = analyse('smoke')
continued_rows = analyse('conversation')
assert all(load(DATA/f'{arm}_summary.json')['gates'].values() for arm in ['smoke','conversation'])


[
  {
    "model": "gpt-6.1-sol",
    "effort": "high",
    "attempted": 5,
    "readable": 5,
    "unique_exact": 5,
    "unique_normalised": 5,
    "unique_post_normalised": 5,
    "cache_read_tokens": 0,
    "input_total_tokens": 54990,
    "cache_read_fraction_all": 0.0,
    "cache_read_fraction_repeats_2_to_5": 0.0,
    "cache_write_tokens": 0,
    "reported_cost_usd": null,
    "seconds": 79.5422785999981,
    "tool_events": 0
  },
  {
    "model": "gpt-6.1-terra",
    "effort": "high",
    "attempted": 5,
    "readable": 5,
    "unique_exact": 5,
    "unique_normalised": 5,
    "unique_post_normalised": 5,
    "cache_read_tokens": 21760,
    "input_total_tokens": 54990,
    "cache_read_fraction_all": 0.3957083106019276,
    "cache_read_fraction_repeats_2_to_5": 0.49463538825240955,
    "cache_write_tokens": 0,
    "reported_cost_usd": null,
    "seconds": 37.52725919999648,
    "tool_events": 0
  },
  {
    "model": "claude-opus-5-5",
    "effort": "high",
    "attempted": 5,
  

[
  {
    "model": "gpt-6.1-sol",
    "effort": "high",
    "attempted": 5,
    "readable": 5,
    "unique_exact": 5,
    "unique_normalised": 5,
    "unique_post_normalised": 5,
    "cache_read_tokens": 22784,
    "input_total_tokens": 58083,
    "cache_read_fraction_all": 0.392266239691476,
    "cache_read_fraction_repeats_2_to_5": 0.4838908357226293,
    "cache_write_tokens": 0,
    "reported_cost_usd": null,
    "seconds": 66.54731990001164,
    "tool_events": 0
  },
  {
    "model": "gpt-6.1-terra",
    "effort": "high",
    "attempted": 5,
    "readable": 5,
    "unique_exact": 5,
    "unique_normalised": 5,
    "unique_post_normalised": 5,
    "cache_read_tokens": 46080,
    "input_total_tokens": 57894,
    "cache_read_fraction_all": 0.7959374028396725,
    "cache_read_fraction_repeats_2_to_5": 0.9825997952917093,
    "cache_write_tokens": 0,
    "reported_cost_usd": null,
    "seconds": 36.22246070000983,
    "tool_events": 0
  },
  {
    "model": "claude-opus-5-5",
    "effort

The following style block produces all saved figures from the response ledgers. Each numbered results section explains the corresponding figure. The tables are descriptive; no interval is justified from one source topic.

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
plt.rcParams.update({'figure.dpi':110,'savefig.dpi':160,'savefig.bbox':'tight',
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':.18,
    'axes.axisbelow':True,'font.size':10,'figure.facecolor':'white'})
FIG = ROOT/'figures'; FIG.mkdir(exist_ok=True)
LABELS = ['GPT-6.1 Sol','GPT-6.1 Terra','Claude Opus 5.5','Claude Sonnet 5.5']
COLORS = ['#245f91','#bf6d30','#39846c','#8265a5']
ARMS = [('smoke','Fresh sessions'),('conversation','One conversation')]
all_records = {arm:[load(p) for p in sorted((DATA/arm).glob('*/*.json'))
                   if '.started.' not in p.name] for arm,_ in ARMS}
fig,axs=plt.subplots(1,2,figsize=(11.5,4.1),sharey=True)
for ax,(arm,title) in zip(axs,ARMS):
    for model,label,color,marker,style in zip(MODELS,LABELS,COLORS,['o','s','^','D'],['-','--',':','-.']):
        rs=sorted([r for r in all_records[arm] if r['requested_model']==model],key=lambda r:r['replicate'])
        ax.plot([r['replicate'] for r in rs],[100*r['cache_read_fraction'] for r in rs],
                marker=marker,linestyle=style,color=color,label=label,linewidth=1.8)
    ax.set(title=title,xlabel='Request / turn (5 posts per model)',xticks=range(1,6),ylim=(-3,104))
axs[0].set_ylabel('Reported input tokens read from cache (%)')
axs[1].legend(loc='center left',bbox_to_anchor=(1,0.5),frameon=False)
fig.suptitle('Cache reuse differs between fresh and continued conversations',fontsize=13)
fig.tight_layout();fig.savefig(FIG/'cache_comparison.png');plt.close(fig)

fig,ax=plt.subplots(figsize=(10,3.8))
x=np.arange(4)
for offset,(arm,title),color in zip([-.19,.19],ARMS,['#245f91','#bf6d30']):
    values=[load(DATA/f'{arm}_summary.json')['models'][i]['unique_post_normalised'] for i in range(4)]
    bars=ax.bar(x+offset,values,.36,label=title,color=color,hatch='//' if arm=='conversation' else '')
    ax.bar_label(bars,labels=[f'{v}/5' for v in values],padding=3)
ax.set(xticks=x,xticklabels=LABELS,ylim=(0,6.1),ylabel='Distinct proposed post bodies',
       title='Every model returned five distinct posts in each condition')
ax.legend(frameon=False,ncol=2,loc='upper center');fig.tight_layout()
fig.savefig(FIG/'distinct_posts.png');plt.close(fig)

features=[('bold_spans','Bold markers'),('emoji_codepoints','Emoji'),('hashtags','Hashtags'),
          ('em_dashes','Em dash'),('response_preface','Post wrapper'),
          ('editorial_note','Editorial note'),('closing_question','Closing question')]
matrix=[];rowlabels=[]
for arm,title in ARMS:
    frame=pd.read_csv(DATA/('formatting.csv' if arm=='smoke' else 'conversation_formatting.csv'))
    for model,label in zip(MODELS,LABELS):
        part=frame[frame.model==model]
        matrix.append([100*(part[key].astype(float)>0).mean() for key,_ in features])
        rowlabels.append(label+' | '+('fresh' if arm=='smoke' else 'continued'))
fig,ax=plt.subplots(figsize=(11,5.5))
im=ax.imshow(matrix,cmap='Blues',vmin=0,vmax=100,aspect='auto')
for i,row in enumerate(matrix):
    for j,value in enumerate(row):
        ax.text(j,i,f'{value/20:.0f}/5',ha='center',va='center',color='white' if value>=60 else '#222222')
ax.set(xticks=range(len(features)),xticklabels=[label for _,label in features],
       yticks=range(len(rowlabels)),yticklabels=rowlabels,
       title='Formatting habits remain visible even when full posts differ')
ax.tick_params(axis='x',rotation=30);ax.grid(False)
fig.colorbar(im,ax=ax,label='Posts containing feature (%)',shrink=.8)
fig.tight_layout();fig.savefig(FIG/'formatting_patterns.png');plt.close(fig)


## 2. Source and exact prompts

The input is [Welcome Le Chonk: Mistral Large 4 on Hugging Face](https://huggingface.co/blog/mistral-large4), published October 29, 2026. The input contains 2,318 words, including code examples. The complete body was used because the task is article-to-post writing, rather than writing from a summary. A longer input also makes a cache test useful without adding artificial padding.

The article describes a model with roughly one trillion total parameters, a small active subset per token, image inputs, a large context window and defensive-security uses. These are source claims, not measurements performed by this notebook. For a reader, the practical choice is which of these facts a post leads with and how it explains their significance.

The source metadata records publication date, retrieval time, extraction steps and hashes in [article.json](data/article.json). The model-date filter is saved in [freshness.json](data/freshness.json): the official [Sol page](https://developers.openai.com/api/docs/models/gpt-6.1-sol) gives an April 30, 2026 cutoff and the [Terra page](https://developers.openai.com/api/docs/models/gpt-6.1-terra) gives June 1, 2026. For Claude, this test uses the release-date fallback: [Opus 5.5](https://www.anthropic.com/claude-opus-5-5) on September 22 and [Sonnet 5.5](https://www.anthropic.com/claude-sonnet-5-5) on September 28. October 29 is later than all four thresholds. Recent publication reduces ordinary training-data overlap; it cannot prove the provider never saw a page. Model IDs are undated, and undocumented later updates remain a limitation.

The title and complete article body were supplied; navigation, author buttons and a duplicate interactive contents widget were removed. Full source text is retained locally in a git-ignored folder because no redistribution licence is assumed. Saved numeric overlap checks allow offline analysis without copying the article into the public repository. The original article's source link and hash remain public.

Initial prompt in both conditions:

```text
Write a LinkedIn post based on the following article.

<article>
{{complete_article_text}}
</article>
```

Every follow-up in the continued condition:

```text
Generate another post.
```

Fresh requests all have the same user-prompt hash. Continued conversations have one article-containing initial turn and four identical short follow-ups. Each model has its own explicit conversation ID; no conversation mixes models or article topics. No request asks for different wording, a new angle, a certain length, a particular audience, or the avoidance of familiar phrases. The model cannot see the audit inventory.

```mermaid
flowchart LR
    H["Harness: one model block at a time<br/>same source, high effort"] --> O["Claude Opus 5.5<br/>claude -p<br/>5 posts and usage"]
    H --> T["GPT-6.1 Terra<br/>codex exec<br/>5 posts and usage"]
    H --> S["Claude Sonnet 5.5<br/>claude -p<br/>5 posts and usage"]
    H --> G["GPT-6.1 Sol<br/>codex exec<br/>5 posts and usage"]
    O --> C["Response caches<br/>20 posts per condition"]
    T --> C
    S --> C
    G --> C
```

The arrows show which CLI receives the source, not messages exchanged between models. Blocks run serially in the saved order Opus, Terra, Sonnet, Sol. The harness waits for each response before the next call. The two patterns below apply separately to each named model.

```mermaid
sequenceDiagram
    participant H as Harness
    participant M as Selected model CLI
    loop Five fresh sessions
        H->>M: New session, full original article prompt
        Note over H: Wait for this response
        M-->>H: New post, usage, new session ID
    end
```

```mermaid
sequenceDiagram
    participant H as Harness
    participant M as Selected model CLI
    H->>M: New conversation, full original article prompt
    M-->>H: Post 1, usage, conversation ID
    loop Turns 2 to 5
        H->>M: Resume explicit ID, Generate another post.
        Note over H: Wait for this response
        M-->>H: New post, usage, same conversation ID
    end
```

The fresh pattern starts over each time. The continued pattern retains earlier user and assistant turns. There are no cross-model handoffs or parallel generation calls in either smoke condition.

## 3. Distinct outputs

Before looking for phrases, check that the harness really asked for new generations. A local response cache returning the first post five times would make a repetition study meaningless. Each response therefore has a full request fingerprint, a separate replicate slot, its raw CLI events and both exact and normalised text hashes. Normalisation changes case and whitespace only; it does not erase substantive wording.

![Every model produced five distinct post bodies in both conditions.](figures/distinct_posts.png)

Each bar counts unique proposed post bodies out of five. All eight bars reach five: no exact or case/whitespace-normalised duplicates were observed in either condition. The raw full responses are also all distinct within each model/condition. These are observed counts, with no uncertainty intervals; five attempts cannot rule out duplicates in a larger run.

All 40 planned responses were readable. There were zero observed tool invocations. The 20 fresh requests have distinct conversation IDs; each continued model block keeps one ID across its five turns. The same article-containing initial prompt hash is used across both conditions. Gates are saved in [fresh summary](data/smoke_summary.json) and [continued summary](data/conversation_summary.json).

Full-response uniqueness is a weak variety test. In the fresh Sol outputs, requests 2 and 4 use the same opening paragraph even though the rest of their posts differs. Word-trigram overlap and word-sequence similarity for every within-model pair are saved in the pairwise tables. These describe reuse, not quality or independent statistical evidence: the ten pairs share the same five posts and one source.

## 4. Provider cache accounting

The chart answers whether subsequent requests reused input computation. Compare each model's points within a panel, then compare the fresh and continued panels. A point at 100% means nearly all reported input was charged/reported as a cache read; it does not mean the answer was replayed. Each point still corresponds to a new generated post.

![Per-request cached-input fractions for fresh sessions and continued conversations.](figures/cache_comparison.png)

The horizontal axis is request number in fresh sessions or turn number in a continued conversation. The vertical axis is reported cached input divided by total input. Lines distinguish models, with markers at the actual requests. There are five observations per model in each panel and no fitted trend or uncertainty interval. Initial requests are not assumed cold because earlier diagnostics and provider cache state may matter.

| Model | Fresh: requests 2-5 cached | Continued: turns 2-5 cached | Fresh total input, 5 requests | Continued total input, 5 turns |
|---|---|---|---|---|
| GPT-6.1 Sol | 0.00% | 48.39% | 54,990 | 58,083 |
| GPT-6.1 Terra | 49.46% | 98.26% | 54,990 | 57,894 |
| Claude Opus 5.5 | 99.97% | 89.43% | 34,795 | 45,342 |
| Claude Sonnet 5.5 | 99.97% | 89.34% | 34,815 | 45,279 |

The table aggregates tokens before dividing, and its percentage columns use requests/turns 2-5 only. In the continued arm, Terra's four follow-ups all hit the cache, while Sol's second and fifth turns hit it and its third and fourth turns did not. This changed their warm read fractions from 49.46% to 98.26% and from 0% to 48.39%, respectively. The same-conversation strategy helped these measured blocks but did not guarantee a hit.

Fresh Claude repeats already read 99.97% of input from cache. Continued turns add previous answers, so they also create new cache entries. Their warm read fractions are 89.43% for Opus and 89.34% for Sonnet. Five continued turns process about 45 thousand total input tokens per Claude model, compared with about 35 thousand for the fresh requests. This is why maximising the percentage and minimising total cost are different objectives.

For Claude, ordinary input, cache reads and cache writes are disjoint: add all three for total input. For Codex the inclusive input counter already contains cached tokens. More subtly, resumed Codex `turn.completed` counters are cumulative across the conversation, while Claude's token usage is per invocation and its reported cost is cumulative. The notebook retains raw counters and differences cumulative counters within each session before summing. Both Codex models' five increments were checked against their CLI rollout's native last-turn counters; [counter scope checks](data/counter_scope_checks.json) contain only those numeric fields.

To encourage reuse, all five requests for a model/article run consecutively, each waiting for the preceding response to finish. The source, optional system line, working directory, effort and available-tool configuration remain stable. Replicate IDs are ledger metadata rather than prompt text. No artificial padding is added. These CLIs expose no equivalent of every API cache-routing or retention control, so the report records observed reuse without claiming to control backend placement. The run is ordered fresh then continued, so it is not a randomised causal experiment on cache strategies.

In [4]:
display(pd.DataFrame([{'model':f['model'],'fresh_warm_read_fraction':f['cache_read_fraction_repeats_2_to_5'],'continued_warm_read_fraction':c['cache_read_fraction_repeats_2_to_5'],'fresh_input':f['input_total_tokens'],'continued_input':c['input_total_tokens']} for f,c in zip(fresh_rows,continued_rows)]))

,model,fresh_warm_read_fraction,continued_warm_read_fraction,fresh_input,continued_input
0,gpt-6.1-sol,0.000000,0.483891,54990,58083
1,gpt-6.1-terra,0.494635,0.982600,54990,57894
2,claude-opus-5-5,0.999713,0.894302,34795,45342
3,claude-sonnet-5-5,0.999713,0.893360,34815,45279


The displayed table uses the same weighted input denominators as the figure and README. Continuation improved reported read fractions for the GPT blocks and added new cache writes for Claude; it did not create a universal higher-hit or lower-cost result.

## 5. Prose and formatting inventory

A fixed inventory of 111 vocabulary items, phrases and structure candidates was saved before the first smoke generation. It includes vocabulary such as `delve`, `underscore` and `pivotal`; transition/signposting phrases such as `here's why` and `why this matters`; endings such as `my takeaway` and `what do you think`; and candidate structures such as a negative contrast followed by a positive claim. The inventory and exact regular expressions are in [prose_lexicon.json](data/prose_lexicon.json).

The papers in Further reading motivate measuring lexical overrepresentation in writing. They do not prove that all 111 entries are overused by these models on LinkedIn. The repository supplies an editorial watchlist, while LinkedIn-specific phrases and structure patterns are stated investigator hypotheses. A hit is not automatically a defect or evidence of AI authorship. For example, a technical article can have a literal need to discuss a model's capabilities or deployment constraints.

Every inventory entry receives a count, including zeroes, in the whole raw response, proposed post, opening sentence, opening paragraph, first 30 words, transition-start windows, non-final paragraph endings, final substantive paragraph, closing sentence and editorial note. Counts include occurrences, posts containing the entry, occurrences per 1,000 words and source occurrences. A phrase appearing in 4 of 5 posts has 80% post incidence, even if one post repeats it twice. Zones can overlap and must not be summed as disjoint parts.

Fresh Sol uses `my takeaway` in 4 of 5 proposed posts, and continued Sol also uses it in 4 of 5. The literal phrase `the opportunity` occurs in all 5 fresh Sol posts and 2 of its continued posts. Neither phrase occurs in the article under the same matching rule. This is useful evidence for the extractor, but one topic cannot establish that either is a general model signature. The complete zero-inclusive tables are [fresh prose counts](data/prose_counts.csv) and [continued prose counts](data/conversation_prose_counts.csv).

The separate exploratory pass counts sequences of two to eight words that recur in at least two posts, whether or not they were anticipated. It tags exact source overlap and labels every result as a one-topic smoke observation. These are not confirmed cross-topic findings. A larger study needs both post incidence and the number of unrelated source topics on which a pattern appears, with discovery topics kept separate from confirmation topics.

![Counts out of five for bold text, emojis, hashtags, em dashes, wrappers, editorial notes and closing questions.](figures/formatting_patterns.png)

Each cell shows how many of five posts or responses contain the named feature. Numeric labels make the chart readable without relying on colour. Formatting inside the proposed post is separated from response wrappers and editorial notes. All 20 fresh posts use hashtags. Every fresh Sonnet response has a literal post wrapper and an editorial note; the continued Sonnet block has no wrappers and only one note. Fresh Opus has five notes; its continued block has none. Those differences show why conversation history is part of the writing condition, not just a cache setting.

The full formatting tables also count words, paragraphs, short lines, blank lines, Markdown bullets, numbered items, emoji-prefixed lines, links, headings and punctuation. An emoji-prefixed line is not automatically a list item. Word and paragraph boundaries are deterministic heuristics, so the examples below check what they actually capture. Structural candidates such as three-part claims, vague attribution or a genuinely unsupported intensifier still need contextual human review before a quality conclusion.

## 6. Worked cases and parser repairs

Examples are selected for an engineering purpose, with stable IDs linking to [all saved outputs](data/posts.md). These are not gold-answer examples: the task has no single correct post. The expected behaviour is a newly generated article-based post, with all surrounding assistant text preserved for analysis. No factual-quality score or human rating was collected.

### Case 1: different posts can share their opening

Fresh Sol requests 2 and 4, IDs `smoke/gpt-6.1-sol/02` and `/04`, have this identical opening paragraph:

> The most interesting thing about Mistral Large 4 isn’t its size. It’s what open weights could mean for defensive cybersecurity.

Their complete post hashes differ, so both pass the new-output gate. Yet the opening did not vary at all. A full-document uniqueness count would conceal this. The measured pattern occurs in these two of five fresh posts, with similar opening framing in other attempts; it remains a one-topic observation.

### Case 2: a short follow-up changes the available context

Continued Sol turn 2, ID `conversation/gpt-6.1-sol/02`, receives only `Generate another post.` after its earlier article and response. It opens:

> A 1M-token context window is impressive. Making that context useful—and practical to serve—is the harder engineering problem.

The first turn leads with defensive security; the second leads with the context window. The new angle is observed behaviour, not proof of what the model intended. The earlier post is now part of the input, so this output cannot be treated as an independent draw from the original article prompt.

### Case 3: follow one cache calculation end to end

Continued Terra turn 2, ID `conversation/gpt-6.1-terra/02`, resumes the same session after the full article prompt. Its new user message is three words. Codex reports cumulative input of 22,321 tokens and cumulative reads of 11,136. Turn 1 had 10,998 input tokens and zero reads. Subtracting yields 11,323 input tokens and 11,136 reads for turn 2: 98.35% reused input. The generated post is new. The saved last-turn counter independently agrees with these differences.

This is why the ledger needs both raw counters and counter scope. Treating 22,321 as turn-2 input would include turn 1 a second time. Treating the three new user words as all input would omit the conversation history.

### Case 4: an editorial note is not a post closing

Fresh Opus request 3, ID `smoke/claude-opus-5-5/03`, adds this note after a horizontal rule:

> *Note:* The article says the model has 49B active parameters, but the model ID is `Mistral-Large-4-1T-A52B`. I left the active-parameter count out of the post. Check the model card before you add it.

The article's prose names 49B active parameters while its checkpoint name contains `A52B`. The note flags an apparent inconsistency; this study does not independently settle the correct count. The relevant observed behaviour is that the response contains both a proposed post and separate advice. All five fresh Opus responses and all five fresh Sonnet responses do this. The note is preserved and counted, but it is not the LinkedIn post's final sentence.

### Case 5: a response wrapper is not an opening hook

Fresh Sonnet request 1, ID `smoke/claude-sonnet-5-5/01`, begins with `**LinkedIn post:**` and then the post. Every fresh Sonnet response includes that wrapper. Counting it as the opening would turn a packaging habit into a misleading result about how the post addresses readers. The extractor removes only the literal wrapper from post-boundary calculations and counts the wrapper separately. Its raw text remains intact.

### Case 6: a warning record is not a model tool call

Terra's first fresh response includes two CLI warning/error events before a successful final response. They concern the experimental host-skill-discovery flag and disabled Code Mode host. The first parser classified every non-text event as a tool event and stopped. Re-reading the same raw JSONL identified zero tool calls and two warnings. The old classification is saved in diagnostics, and a regression fixture now distinguishes warning records from actual tool events.

No unreadable final post remained among the 40 planned outputs. One earlier Codex launch failed before generation because strict configuration rejected a field. One earlier Opus diagnostic exposed built-in skill inventory labels. These are engineering failures, not wrong answers about the article. They are preserved separately, with no content regenerated merely because its wording was repetitive or inconvenient.

The location parser was amended after seeing wrappers, notes and trailing emojis. The original location tables are retained under diagnostics. The summary's distinct-post and cache results do not depend on treating an editorial note as a post ending, but claims about opening/closing frequencies must use the corrected boundary definitions. [Protocol history](data/PROTOCOL.md) gives the amendment timing and both classifications.


## 7. What this test can establish

The harness can obtain five distinct post bodies with either fresh sessions or the requested continued-conversation pattern, and it can extract provider-reported cache information with the correct counter scope. This supports using the continued pattern for the requested larger study: one separate conversation for each model/article, one full article prompt and four short follow-ups. It does not establish that continuation is always the cheapest strategy or always produces more varied writing.

One topic is too little to separate topic vocabulary from model preferences. These 40 posts must not support a family ranking. A phrase shared across two models may arise from the article, the LinkedIn task, their CLI scaffolding or prior posts. Two models per family would also be too few to generalise to every model that vendor makes.

The fresh and continued arms ran in that order, with cache state carried from diagnostics and earlier requests. Differences cannot be assigned causally to conversation state alone. Follow-up outputs are conditional on their earlier posts. In the main study the article/conversation should be the unit used for resampling uncertainty; the five turns are not five unrelated articles.

The source's apparent parameter-count mismatch may have encouraged the editorial notes. Another article could yield none. Similarly, emoji and hashtag habits may change with topic. A source-normalised count helps describe copying, but it cannot prove that an overlapping phrase was copied, or that a non-overlapping phrase is inherently model-specific.

The CLIs add provider/runtime context that cannot all be removed. No observed tool use, empty Claude executable-tool catalogues and supported disable switches are real checks; an identical pure system prompt is not. Requested model IDs are undated. No human readability study, blinded quality assessment, factual score or LinkedIn engagement test was performed.

For the main study, keep the initial and follow-up wording fixed and retain the 111-entry inventory independently of newly discovered phrases. Measure exact expressions, structure candidates and formatting separately. Confirm discovered cross-topic patterns on held-out topics. Replace this smoke topic in the planned main sample, since it has already been inspected during harness development.

## 8. Spend and reproduction

The primary saved sample contains 40 planned generations, split evenly between the two conditions, plus separate diagnostics. Claude's CLI-reported list-price equivalent for the 20 included Claude responses is $0.439865. The ten fresh responses account for $0.227025 and the ten continued responses for $0.212840. Codex exposes token counters but no comparable dollar total in these records, so the full dollar equivalent is unavailable. These values are not an invoice or proof of extra subscription charges.

Earlier diagnostic requests have their own saved usage. An Opus request was interrupted while adapting to the requested follow-up rule, before its final provider usage was captured; that cost is unknown. Do not present the sum of known CLI values as the total spend. There are no paid judge calls. [The protocol](data/PROTOCOL.md) records request history, configuration rejection, parser repairs and the superseded conversation condition.

Key files:

- [Executed notebook](40_linkedin_writing_habits.ipynb), with all helpers embedded and default offline replay.
- [Standalone CLI collector](smoke_study.py) and [regression fixtures](test_smoke_study.py).
- [All generated responses](data/posts.md), including separate preface/body/note views.
- [Fresh cache ledger](data/provider_cache_usage.csv) and [continued cache ledger](data/conversation_provider_cache_usage.csv).
- [Fresh prose counts](data/prose_counts.csv) and [continued prose counts](data/conversation_prose_counts.csv), including zeroes.
- [Source provenance](data/article.json), [frozen inventory](data/prose_lexicon.json), [counter-scope verification](data/counter_scope_checks.json) and [protocol history](data/PROTOCOL.md).

From this folder, replay and test without model calls:

```powershell
python -m unittest -v test_smoke_study.py
python smoke_study.py analyse
python smoke_study.py analyse-conversation
python -m nbconvert --to notebook --execute --inplace 40_linkedin_writing_habits.ipynb
```

The notebook defaults to saved evidence and needs Python, nbformat/nbconvert, pandas, NumPy and matplotlib. It imports no helper from another repository folder. The standalone collector also uses requests and Beautiful Soup when preparing a new source download; these packages are not used for model calls. Source-overlap hashes and counts permit replay even when the git-ignored full article is absent.

Live CLI collection is deliberately explicit:

```powershell
python smoke_study.py collect --repeats 5 --max-new 20
python smoke_study.py conversation --repeats 5 --max-new 20
```

Existing slots replay; they do not generate again. Conversation turn IDs and source/prompt fingerprints prevent an output from one repeat substituting for another. A new collection needs the identical frozen source hash or a separately labelled protocol/source version. Inspect existing evidence before deleting any slot: removing a slot in the middle of a continued conversation and resuming from a later-mutated session would change history. Source text is not redistributed here. Re-downloads may change and must not silently replace the frozen input.

Local regression fixtures test disjoint Claude cache fields, unknown counters, resumed Codex cumulative accounting, warning/tool distinctions, output boundaries, final emojis and separate replicate slots. Live smoke generation tests the real process invocation, persistence, explicit resume IDs and response schemas. No API key is needed for offline reproduction.

In [5]:
reported = sum(r['reported_cost_usd'] or 0 for arm in all_records.values() for r in arm)
print(f'Included Claude CLI list-price equivalent: ${reported:.6f}')
print('Codex dollar total: unavailable. Interrupted diagnostic cost: unknown. Total spend cannot be stated.')

Included Claude CLI list-price equivalent: $0.439865
Codex dollar total: unavailable. Interrupted diagnostic cost: unknown. Total spend cannot be stated.


## 9. Future work and further reading

Links checked November 1, 2026 (Toronto local date).

### Future work

- Run the requested 20-topic study with four models at high effort, one conversation per model/article and five posts per conversation. Measure cross-topic phrase incidence, within-conversation reuse and formatting. The repeated Sol openings and wrapper changes motivate looking beyond full-text hashes.
- Randomise the order of fresh and continued cache blocks across several articles and repeat at different times. Measure cache reads, writes, ordinary input and cost equivalents. The current intermittent Codex hits and ordered arms cannot establish a stable causal saving.
- Test matched authored articles with and without an apparent numerical inconsistency. Measure editorial-note rates and factual support. Ten fresh Claude notes versus no fresh GPT notes motivate asking whether the source mismatch drives the response format.
- Add a blinded human review of a preselected, topic-balanced main-study subset. Measure factual faithfulness, clarity, specificity and readiness to publish separately from repetition. This smoke has no quality ground truth or engagement outcome.

### Further reading

- [Delving into LLM-assisted writing in biomedical publications through excess vocabulary](https://arxiv.org/abs/2406.07016) (Kobak et al., 2025). Motivates measuring vocabulary frequency against an explicit comparison rather than treating a word as an authorship detector.
- [Why Does ChatGPT "Delve" So Much? Exploring the Sources of Lexical Overrepresentation in Large Language Models](https://arxiv.org/abs/2412.11385) (Juzek and Ward, 2024). Examines overrepresented vocabulary; its scientific-writing setting does not by itself establish LinkedIn habits.
- [Prompt caching](https://developers.openai.com/api/docs/guides/prompt-caching) (OpenAI, accessed 2026). Explains prefix reuse, minimum lengths and usage fields; API controls must not be assumed available through Codex.
- [Prompt caching](https://platform.claude.com/docs/en/build-with-claude/prompt-caching) (Anthropic, accessed 2026). Explains cache reads/writes and growing conversation prefixes, including why denominator definitions matter.
